In [1]:
import numpy as np 
import pandas as pd

In [2]:
def generate_total_seqs_per_donor(load_path):
    '''
    All data that goes here must be before redundant seqs are removed
        cdrh3_seqs_by_donor = no seqs were removed.
    '''
    import numpy as np
    donors_list = np.load(f'{load_path}donors_list.npy')
    donors_list_unique = np.unique(donors_list)
    total_seqs_per_donor = []
    donor_name = []
    for donors in range(len(donors_list_unique)):
        donor = donors_list_unique[donors]

        # get the number of sequences from each donor in the full OAS database
        counter = []
        with open(f"/Users/ssolieva/Desktop/Kulp_lab/projects/OAS_database_searches/donor_counts/{donor}.out", "r") as fd: # import the results file
            file_contents = fd.read().splitlines() # read in the results
            for i in range(len(file_contents)):
                counter.append(int(file_contents[i]))
            #print(f'{donor}, Number of sequences in OAS database: {sum(counter)}')
            n_seqs_donor = sum(counter)
        fd.close()
        total_seqs_per_donor.append(n_seqs_donor)
        donor_name.append(donor)
    return donor_name, total_seqs_per_donor

In [3]:
def save_out_data(search_name):
    '''
    Load in all sequences. 
    Save all data as CSV files.
    '''
    load_path=f'/Users/ssolieva/Desktop/github_repo/Q23_paper/OAS_database_searches/PG9/{search_name}/data/parsed_logfile/' # path to where the search results are (logfile).  
    donor_name, n_total_seqs = generate_total_seqs_per_donor(load_path) # load in the donor counts and list of donors (without redudant sequences removed)
    
    all_hits = np.load(f'{load_path}cdrh3_seqs_by_donor.npy', allow_pickle=True) # this includes redundant sequences. 
    n_all_hits = [] # number of all hits, including redundanct sequences, per donor. 
    for i in range(len(all_hits)): # iterate through all donors. 
        n_all_hits.append(int(len(all_hits[i]))) # only save number of sequences. 
        
    collapsed_hits = np.load(f'{load_path}redundant_seqs_removed_within_donors_cdrh3_final.npy', allow_pickle=True) # load in the data after redudant sequences per donor have been removed. 
    n_collapsed_hits = [] # number of non-redundant sequences per donor. 'collapsed' = non-redundant. 
    for i in range(len(collapsed_hits)): # iterate through all donors. 
        n_collapsed_hits.append(int(len(collapsed_hits[i]))) # save the number of sequences. 
    
    n_seqs_removed = [] # the number of redundant sequences per donor. 
    for i in range(len(n_all_hits)): # iterate through all donors. 
        n_seqs_removed.append(int(n_all_hits[i]-n_collapsed_hits[i])) # save the difference in number of sequences. 
        
    n_total_seqs_minus_n_redundant_hits = [] # the number of total sequences minus known redundant sequences (from hits list) per donor. Needed for conservative frequency calculation. 
    for i in range(len(n_all_hits)): 
        n_total_seqs_minus_n_redundant_hits.append(int(n_total_seqs[i]-n_seqs_removed[i]))
    
    original_frequency_estimate=[] # the original frequency estimate per donor, includes all redundant sequences. 
    for i in range(len(n_all_hits)):
        original_frequency_estimate.append(float((n_all_hits[i]/n_total_seqs[i])*1000000))
        
    conservative_frequency_estimate=[]
    for i in range(len(n_all_hits)): 
        conservative_frequency_estimate.append(float((n_collapsed_hits[i]/n_total_seqs_minus_n_redundant_hits[i])*1000000))
        
    data = {
        "donor_name":       donor_name,
        "n_total_seqs":     n_total_seqs,
        "n_all_hits":       n_all_hits,
        "n_nonredundant_hits": n_collapsed_hits, 
        "n_redundant_hits":    n_seqs_removed, 
        "n_total_seqs_minus_n_redundant_hits" : n_total_seqs_minus_n_redundant_hits,
        "original_frequency_estimate":original_frequency_estimate,
        "conservative_frequency_estimate": conservative_frequency_estimate
    }
    
    df = pd.DataFrame(data)    
    df.to_csv(f'PG9_{search_name}.csv')

In [4]:
# csv files saved to q23-manuscript/repertoire-frequency-analysis/OAS-searches/data 
search_names = ['search1', 'search2', 'search3', 'search4', 'search5', 'search6', 'search7']

for search_name in search_names:
    save_out_data(search_name)